In [1]:
# import packages
import cv2
import glob
import numpy as np
import rawpy
import gc

# lens correction helpers
import json
import shutil
import subprocess
import lensfunpy

In [2]:
# Force OpenCV to run entirely on the CPU and completely disable OpenCL acceleration
# addresses an issue I was having (thanks Gemini)
cv2.ocl.setUseOpenCL(False)

In [ ]:
# find all sections in the raw data folder. Assumes folder names are structured as "raw_data/<section_name>"
sampleID = sorted(glob.glob("raw_data/*"))

# remove the leading "raw_data/" from each folder name
sampleID = [folder.replace("raw_data/", "") for folder in sampleID]
print(sampleID)

# NOTE: Alternatively, you can mannually define a list of core sections to run as below.
sampleID = ["ALHIC2502-47"]

['ALHIC2502-102-A', 'ALHIC2502-102-D', 'ALHIC2502-103', 'ALHIC2502-24', 'ALHIC2502-47', 'ALHIC2502-86']


In [ ]:
# set params


# set filepaths
raw_data_dir = "raw_data/"
output_dir = "image_outputs/"

# confidence threshold and scaling - note that you might need to tune these to avoid things blowing up!!!
pano_conf_thresh_lowres = .85
pano_conf_thresh_highres = .9
scale_factor = 0.15  # Downsample to 15% size for feature matching

# lens correction
lens_correction = True

In [5]:
# Optional fallback for older Nikon metadata that may omit LensModel.
# Fill this mapping if profile lookup fails with LensID-only files.
LENS_ID_TO_MODEL = {
    # "32 54 6A 6A 24 24 35 02": "AF-S VR Micro-Nikkor 105mm f/2.8G IF-ED",
}

EXIFTOOL_AVAILABLE = shutil.which("exiftool") is not None
if not EXIFTOOL_AVAILABLE:
    print("exiftool is not installed. Install with: brew install exiftool")

LENS_DB = lensfunpy.Database() if lensfunpy is not None else None
lens_correction_ready = EXIFTOOL_AVAILABLE and (LENS_DB is not None)

LENS_META_CACHE = {}
LENS_PROFILE_CACHE = {}
LENS_MAP_CACHE = {}

def _to_float(v, default):
    try:
        return float(v)
    except Exception:
        return float(default)

def read_nef_meta(path):
    if path in LENS_META_CACHE:
        return LENS_META_CACHE[path]

    tags = [
        "Make", "Model", "LensModel", "Lens", "LensID",
        "FocalLength", "FNumber", "FocusDistance"
    ]
    cmd = ["exiftool", "-j", "-n"] + [f"-{t}" for t in tags] + [str(path)]
    out = subprocess.check_output(cmd, text=True)
    d = json.loads(out)[0]

    lens_model = d.get("LensModel") or d.get("Lens")
    if not lens_model:
        print(f"  [Info] Using fallback lens model for {path}")
        lens_model = LENS_ID_TO_MODEL.get(d.get("LensID"))

    meta = {
        "make": d.get("Make"),
        "model": d.get("Model"),
        "lens_model": lens_model,
        "lens_id": d.get("LensID"),
        "focal": _to_float(d.get("FocalLength", 50.0), 50.0),
        "aperture": _to_float(d.get("FNumber", 8.0), 8.0),
        "distance": _to_float(d.get("FocusDistance", 10.0), 10.0),
    }
    LENS_META_CACHE[path] = meta
    return meta

def _find_profile(meta):
    key = (meta["make"], meta["model"], meta["lens_model"] )
    if key in LENS_PROFILE_CACHE:
        return LENS_PROFILE_CACHE[key]

    if not key[0] or not key[1] or not key[2]:
        LENS_PROFILE_CACHE[key] = (None, None)
        return None, None

    cams = LENS_DB.find_cameras(meta["make"], meta["model"], loose_search=True)
    if not cams:
        LENS_PROFILE_CACHE[key] = (None, None)
        return None, None
    cam = cams[0]

    lenses = LENS_DB.find_lenses(cam, None, meta["lens_model"], loose_search=True)
    if not lenses:
        lenses = LENS_DB.find_lenses(cam, meta["make"], meta["lens_model"], loose_search=True)

    if not lenses:
        LENS_PROFILE_CACHE[key] = (None, None)
        return None, None

    lens = lenses[0]
    LENS_PROFILE_CACHE[key] = (cam, lens)
    return cam, lens

def apply_lens_correction_bgr(path, bgr):
    if not lens_correction_ready:
        return bgr

    try:
        meta = read_nef_meta(path)
    except Exception as exc:
        print(f"  [Warning] Metadata read failed for {path}: {exc}")
        return bgr

    cam, lens = _find_profile(meta)
    if cam is None or lens is None:
        return bgr

    h, w = bgr.shape[:2]
    # Cache by lens identity + image size + focal length to avoid exploding map cache.
    map_key = (
        meta["make"], meta["model"], meta["lens_model"], w, h,
        round(meta["focal"], 3)
    )

    if map_key not in LENS_MAP_CACHE:
        mod = lensfunpy.Modifier(lens, cam.crop_factor, w, h)
        mod.initialize(meta["focal"], meta["aperture"], meta["distance"] )
        coords = mod.apply_geometry_distortion()
        map_x = coords[:, :, 0].astype(np.float32)
        map_y = coords[:, :, 1].astype(np.float32)
        LENS_MAP_CACHE[map_key] = (map_x, map_y)

    map_x, map_y = LENS_MAP_CACHE[map_key]
    corrected = cv2.remap(
        bgr, map_x, map_y,
        interpolation=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT
    )
    return corrected

if lens_correction and not lens_correction_ready:
    print("Lens correction enabled but dependencies are missing. Running without correction.")

In [6]:
# lens correction helpers (Lensfun + EXIF metadata from NEF)
import json
import shutil
import subprocess

try:
    import lensfunpy
except ImportError:
    lensfunpy = None
    print("lensfunpy not installed. Install with: pip install lensfunpy")

# Optional fallback for older Nikon files missing LensModel.
# Add a mapping once you confirm the exact lens string.
LENS_ID_TO_MODEL = {
    # "32 54 6A 6A 24 24 35 02": "AF-S VR Micro-Nikkor 105mm f/2.8G IF-ED",
}

LENS_META_CACHE = {}
LENS_PROFILE_CACHE = {}
LENS_MAP_CACHE = {}
LENS_DB = lensfunpy.Database() if lensfunpy is not None else None


def _to_float(v, default):
    try:
        return float(v)
    except Exception:
        return float(default)


def read_nef_meta(path):
    if path in LENS_META_CACHE:
        return LENS_META_CACHE[path]

    tags = [
        "Make", "Model", "LensModel", "Lens", "LensID",
        "FocalLength", "FNumber", "FocusDistance"
    ]
    cmd = ["exiftool", "-j", "-n"] + [f"-{t}" for t in tags] + [str(path)]
    out = subprocess.check_output(cmd, text=True)
    d = json.loads(out)[0]

    lens_model = d.get("LensModel") or d.get("Lens")
    if not lens_model:
        lens_model = LENS_ID_TO_MODEL.get(d.get("LensID"))

    meta = {
        "make": d.get("Make"),
        "model": d.get("Model"),
        "lens_model": lens_model,
        "lens_id": d.get("LensID"),
        "focal": _to_float(d.get("FocalLength", 50.0), 50.0),
        "aperture": _to_float(d.get("FNumber", 8.0), 8.0),
        "distance": _to_float(d.get("FocusDistance", 10.0), 10.0),
    }
    LENS_META_CACHE[path] = meta
    return meta


def _find_profile(meta):
    if LENS_DB is None:
        return None, None

    key = (meta["make"], meta["model"], meta["lens_model"] )
    if key in LENS_PROFILE_CACHE:
        return LENS_PROFILE_CACHE[key]

    if not meta["make"] or not meta["model"] or not meta["lens_model"]:
        LENS_PROFILE_CACHE[key] = (None, None)
        return None, None

    cams = LENS_DB.find_cameras(meta["make"], meta["model"], loose_search=True)
    if not cams:
        LENS_PROFILE_CACHE[key] = (None, None)
        return None, None
    cam = cams[0]

    lenses = LENS_DB.find_lenses(cam, None, meta["lens_model"], loose_search=True)
    if not lenses:
        lenses = LENS_DB.find_lenses(cam, meta["make"], meta["lens_model"], loose_search=True)
    if not lenses:
        LENS_PROFILE_CACHE[key] = (None, None)
        return None, None

    lens = lenses[0]
    LENS_PROFILE_CACHE[key] = (cam, lens)
    return cam, lens


def apply_lens_correction_bgr(path, bgr):
    # No-op if dependencies are missing.
    if lensfunpy is None or shutil.which("exiftool") is None:
        return bgr

    try:
        meta = read_nef_meta(path)
    except Exception:
        return bgr

    cam, lens = _find_profile(meta)
    if cam is None or lens is None:
        return bgr

    h, w = bgr.shape[:2]
    # Cache by lens identity + image size + focal length to avoid exploding map cache.
    map_key = (
        meta["make"], meta["model"], meta["lens_model"], w, h,
        round(meta["focal"], 3)
    )

    if map_key not in LENS_MAP_CACHE:
        mod = lensfunpy.Modifier(lens, cam.crop_factor, w, h)
        mod.initialize(meta["focal"], meta["aperture"], meta["distance"] )
        coords = mod.apply_geometry_distortion()
        map_x = coords[:, :, 0].astype(np.float32)
        map_y = coords[:, :, 1].astype(np.float32)
        LENS_MAP_CACHE[map_key] = (map_x, map_y)

    map_x, map_y = LENS_MAP_CACHE[map_key]
    corrected = cv2.remap(
        bgr, map_x, map_y,
        interpolation=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT
    )
    return corrected

In [7]:

if lens_correction and lens_correction_ready:
    print("Lens correction is enabled.")
elif lens_correction:
    print("Lens correction requested but unavailable. Continuing without correction.")

# Loop through folders safely
for sample in sampleID:
    print("\n" + "="*50)
    print(f"Running images from: {sample}")
    print("="*50)

    if lens_correction and lens_correction_ready and LENS_MAP_CACHE:
        # Prevent growth of remap matrices across samples.
        LENS_MAP_CACHE.clear()
        gc.collect()

    # 1. Locate files dynamically inside the target sample folder
    nef_paths = sorted(glob.glob(f"{raw_data_dir}/{sample}/*.NEF"))
    print(f" Found {len(nef_paths)} NEF files in folder '{sample}'.")
    
    if len(nef_paths) == 0:
        print("  [Warning] No files found. Skipping this folder.")
        continue

    # Step 1: Extract low-res thumbnails to calculate alignment geometry
    print(" Step 1: Extracting low-res thumbnails to calculate alignment...")
    low_res_images = []
    for path in nef_paths:
        with rawpy.imread(path) as raw:
            rgb = raw.postprocess(use_camera_wb=True, half_size=True)
        
        bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        if lens_correction and lens_correction_ready:
            bgr = apply_lens_correction_bgr(path, bgr)
        
        # Calculate bounds based on target scale factor
        h, w = bgr.shape[:2]
        target_w = int(w * scale_factor * 2)  # Compensates for fast half_size decode
        target_h = int(h * scale_factor * 2)
        
        small_img = cv2.resize(bgr, (target_w, target_h), interpolation=cv2.INTER_AREA)
        low_res_images.append(small_img)

    # Step 2: Test alignment geometry using flat grid SCANS mode (value 1)
    print(" Step 2: Aligning 2D zig-zag grid structure...")
    stitcher = cv2.Stitcher_create(1)
    stitcher.setPanoConfidenceThresh(pano_conf_thresh_lowres)    # set lower confidence threshold - neccisary for some rows
    status, low_res_panorama = stitcher.stitch(low_res_images)

    # Free up memory allocated to thumbnails immediately
    del low_res_images
    gc.collect()

    # check status
    if status != cv2.Stitcher_OK:
        print(f"  [Error] Stitching geometry failed! Error code: {status}")
        print("  Tip: Verify that the frame-to-frame step overlap is at least 20-30%.")
        continue # Skip to next core sample instead of crashing the whole batch script
    print("  Grid layout successfully calculated!")

    # Step 3: High resolution processing with safe RAM footprints
    print(" Step 3: Generating final high-res grid composition...")
    stitcher_high = cv2.Stitcher_create(1)
    stitcher_high.setRegistrationResol(0.6) 
    stitcher_high.setPanoConfidenceThresh(pano_conf_thresh_highres) # <- reduce confidence threshold to avoid missing matches
    high_res_images = []
    for path in nef_paths:
        with rawpy.imread(path) as raw:
            # Full quality decode
            rgb = raw.postprocess(use_camera_wb=True, half_size=False)
        bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        if lens_correction and lens_correction_ready:
            bgr = apply_lens_correction_bgr(path, bgr)
        high_res_images.append(bgr)
    print("  Assembling canvas matrices...")
    status, high_res_panorama = stitcher_high.stitch(high_res_images)

    # Instantly wipe the high-res list elements out of memory cache
    del high_res_images
    gc.collect()

    # check status, save file
    if status == cv2.Stitcher_OK:
        # Standardized file pathing structure
        output_path = f"{output_dir}/{sample}_stitched_output.jpg"
        cv2.imwrite(output_path, high_res_panorama)
        print(f"  Success! Stitched grid saved to: {output_path}")
        
        # Completely drop panorama matrix representation to reset memory baseline
        del high_res_panorama
        gc.collect()
    else:
        print(f"  [Error] Highmultip resolution layout composition processing failed: {status}")

Lens correction is enabled.

Running images from: ALHIC2502-86
 Found 52 NEF files in folder 'ALHIC2502-86'.
 Step 1: Extracting low-res thumbnails to calculate alignment...
 Step 2: Aligning 2D zig-zag grid structure...
  Grid layout successfully calculated!
 Step 3: Generating final high-res grid composition...
  Assembling canvas matrices...
  Success! Stitched grid saved to: image_outputs//ALHIC2502-86_stitched_output.jpg
